# ocean: zonal

zonal ocean data

**coordinate**
* tavg-ol-hyb-sea
    - tavg: time average
    - ol: ocean levels
    - hyb: zonal mean
    - sea: ocean domain

In [ ]:
## Import libraries
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cmocean
import sys
import os
import glob
from IPython.display import HTML, display

sys.path.append(os.getcwd())
from utils import load_grid_vertex, read_variables, read_compound_names

In [ ]:
# parameters for the cmorized data
cmorout=''                  # root for cmorized data, e.g., '/scratch/$USER/cmorout'
source_id      = ''         # model name, e.g., 'NorESM3-LM'
experiment_id  = ''         # experiment name, e.g., 'historical', 'ssp585', 'piControl'
variant_label  = ''         # variant label, e.g., 'r1i1p1f1'
grid_label     = ''         # grid label, e.g., 'gn', 'gr', 'g999'
version        = ''         # version, e.g., 'v20260601'

In [ ]:
# data path
data_path = os.path.join(cmorout, source_id, experiment_id, version)

# load grid
grid_file = 'data/grid.nc'
lat, lon, clat, clon = load_grid_vertex(grid_file)
with xr.open_dataset(grid_file) as ds:
    pmask = ds['pmask']
    parea = ds['parea']

parea = parea.rename({'y': 'j', 'x': 'i'})
pmask = pmask.rename({'y': 'j', 'x': 'i'})
# load methods for plotting and set defaults
methods =read_variables('data/methods.txt')
#print(methods.keys())

---
**List of datasets:** \
(datasets which are not presented/cmorized have no link.)

In [ ]:
# load compound names

coords = ('tavg-ol-hyb-sea','tavg-rho-hyb-sea','tavg-u-hyb-sea')
cnames = read_compound_names('data/variables.nml')
# examples of compound names:
# cnames = ['ocean.tos.tavg-u-hxy-sea.mon.glb', 'ocean.ficeberg.tavg-u-hxy-sea.mon.glb']
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    
    if realm != 'ocean' or coord not in coords:
        continue
    else:
        data_file = var+'_'+coord+'_*_'+grid_label+'_'+source_id+'_'+experiment_id+'_'+variant_label+'_*.nc'
        if not glob.glob(os.path.join(data_path, data_file)):
            print(cname)
            continue
        else:
            display(HTML(f'<a href="#{cname}">{cname}</a>'))


---
**Datasets validated:**

In [ ]:
 # loop through compound names and plot
for cname in cnames:
    mth_cmap = 'mpl.colormaps["seismic"]'
    if cname not in methods.keys():
        print(f"{cname} not found in methods.txt, using default methods for plotting.")
    else:
        if methods[cname] is not None:

            if 'cmap' in methods[cname].keys():
                mth_cmap = methods[cname]['cmap']

    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]

    if realm != 'ocean' or coord not in coords:
        continue

    data_file = var+'_'+coord+'_*_'+grid_label+'_'+source_id+'_'+experiment_id+'_'+variant_label+'_*.nc'

    if not glob.glob(os.path.join(data_path, data_file)):
        continue

    #with xr.open_mfdataset(os.path.join(data_path, data_file)) as ds:
    data_file = glob.glob(os.path.join(data_path, data_file))[0]
    with xr.open_dataset(os.path.join(data_path, data_file)) as ds:
        if var in ds:
            data = ds[var]
        else:
            continue

    data2d = data 

    # Scale
    vmin, vmax = data2d.quantile([0.001, 0.999], dim=None).values
    vmax = max(abs(vmin),abs(vmax))
    vmin = -vmax
    # Define the discrete levels (boundaries) and the corresponding colors
    n_levels = 20
    levels = np.linspace(vmin, vmax, n_levels+1)
    colors = eval(mth_cmap)(np.linspace(0,1,n_levels))
    cmap = ListedColormap(colors)

    # Create a BoundaryNorm to map data to discrete color indices
    norm = BoundaryNorm(levels, ncolors=len(cmap.colors), clip=False)

    display(HTML(f'<div id="{cname}"></div>'))
    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data.long_name} ({data.units})')
    print(f'NorESM->CMOR: {data.attrs["original_name"]} -> {var}')
    if 'history' in data.attrs:
        print(f'history: {data.attrs["history"]}')
    if 'comment' in data.attrs:
        print(f'comment: {data.attrs["comment"]}')

    nlines = data.basin.size
    sector = ds['sector'].data
    fig, axs = plt.subplots((nlines+1)//2, 2, figsize=(11.69, 8.27), dpi=96, sharex=True, sharey=False)
    ax = axs.flatten()
    for n in range(nlines):
        if any(dim in data.dims for dim in ['lev', 'rho']):
            if vmin == vmax:
                data.isel(basin=n).mean(dim='time').plot(ax=ax[n], cmap=cmap, bar_kwargs={"extend":"both", "label": f"{data.name} ({data.units})"})
            else:
                data.isel(basin=n).mean(dim='time').plot(ax=ax[n], cmap=cmap, norm=norm, cbar_kwargs={"extend":"both", "label": f"{data.name} ({data.units})"})
            ax[n].invert_yaxis()
        else:
            data.isel(basin=n).mean(dim='time').plot(ax=ax[n])
            ax[n].set_ylabel(data.units)
            
        ax[n].set_title(f'{sector[n].astype(str).strip()}')
        ax[n].set_xlabel('')
        ax[n].grid()

    # Turn off x‑axis ticks for all subplots
    for ax in axs.flatten():
        ax.xaxis.set_tick_params(labelbottom=False, bottom=False)

    # Turn on x‑axis ticks only for the bottom row (last two axes)
    for ax in axs[-1,:]:
        ax.xaxis.set_tick_params(labelbottom=True, bottom=True)
        ax.set_xlabel(data.lat.long_name)

    fig.suptitle(f"{data.long_name}", fontsize=16)

    plt.tight_layout()
    plt.show()

    del data
    del ax, axs, fig 